# 2.6 — Conversational RAG

Basic RAG answers one question at a time. **Conversational RAG** adds chat history so the model can handle follow-up questions.

**Problem with basic RAG:**
```
User: How many leave days do I get?
Bot:  20 days of annual leave.
User: What about sick leave?         ← 'what about' needs prior context
Bot:  ??? (doesn't know what 'what about' refers to)
```

**Solution:** Rephrase follow-up questions using chat history before retrieving.

In [ ]:
!pip install langchain langchain-ollama langchain-community chromadb --quiet

## Step 1 — Set Up the Vector Store

In [ ]:
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

docs = [
    Document(page_content='All full-time employees receive 20 days of annual leave per year.'),
    Document(page_content='Sick leave is up to 10 days per year with a medical certificate.'),
    Document(page_content='Parental leave is 16 weeks fully paid for primary caregivers.'),
    Document(page_content='Leave requests must be submitted at least 2 weeks in advance.'),
    Document(page_content='Employees may work remotely up to 3 days per week.'),
    Document(page_content='Remote workers must be available during core hours: 10am to 3pm.'),
    Document(page_content='All remote work equipment is provided by the company.'),
    Document(page_content='Health insurance is provided for all full-time employees and their immediate family.'),
    Document(page_content='A gym membership subsidy of $50 per month is available.'),
    Document(page_content='Employees receive a $1,000 annual learning and development budget.'),
    Document(page_content='Standard working hours are 9am to 5pm, Monday to Friday.'),
    Document(page_content='Overtime must be pre-approved and compensated at 1.5x the hourly rate.'),
]

embeddings = OllamaEmbeddings(model='llama3.2')
vectorstore = Chroma.from_documents(docs, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
print('Vector store ready.')

## Step 2 — Build the Conversational RAG Chain

We use two prompts:
1. **Rephrase prompt** — rewrite the follow-up question as a standalone question
2. **Answer prompt** — answer using retrieved context

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableBranch
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatOllama(model='llama3.2', temperature=0)

# Prompt 1: rephrase follow-up question using chat history
rephrase_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Given the chat history and a follow-up question, rephrase the follow-up as a standalone question. Return only the rephrased question.'),
    MessagesPlaceholder('chat_history'),
    ('human', '{question}'),
])

# Prompt 2: answer the question using retrieved context
answer_prompt = ChatPromptTemplate.from_template("""
You are a helpful HR assistant. Answer using only the context provided.
If the information is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

print('Prompts ready.')

## Step 3 — Agent Function with Chat History

In [ ]:
def conversational_rag(question: str, chat_history: list) -> str:
    """
    chat_history: list of {'role': 'user'/'assistant', 'content': str}
    """
    # Convert history dicts to LangChain message objects
    lc_history = []
    for msg in chat_history:
        if msg['role'] == 'user':
            lc_history.append(HumanMessage(content=msg['content']))
        else:
            lc_history.append(AIMessage(content=msg['content']))

    # If there's history, rephrase the question as standalone
    if lc_history:
        rephrase_chain = rephrase_prompt | llm | StrOutputParser()
        standalone_question = rephrase_chain.invoke({
            'chat_history': lc_history,
            'question': question
        })
        print(f'  [Rephrased] {standalone_question}')
    else:
        standalone_question = question

    # Retrieve relevant chunks
    context_docs = retriever.invoke(standalone_question)
    context = format_docs(context_docs)

    # Generate answer
    answer_chain = answer_prompt | llm | StrOutputParser()
    answer = answer_chain.invoke({'context': context, 'question': standalone_question})

    return answer

print('Function ready.')

## Step 4 — Test Conversational RAG

In [ ]:
chat_history = []

# Simulate a multi-turn conversation
questions = [
    'How many leave days do employees get?',
    'What about sick leave?',             # follow-up
    'And for new parents?',               # another follow-up
    'How far in advance do I need to apply?',  # another follow-up
]

for question in questions:
    print(f'User: {question}')
    answer = conversational_rag(question, chat_history)
    print(f'Bot:  {answer}')
    print()

    # Update history
    chat_history.append({'role': 'user', 'content': question})
    chat_history.append({'role': 'assistant', 'content': answer})

## Step 5 — Gradio Chat UI

In [ ]:
import gradio as gr

def chat(user_message, history):
    answer = conversational_rag(user_message, history)
    history.append({'role': 'user', 'content': user_message})
    history.append({'role': 'assistant', 'content': answer})
    return '', history

with gr.Blocks(title='HR Policy Q&A') as demo:
    gr.Markdown('# HR Policy Q&A Bot\nAsk me anything about the employee handbook.')
    chatbot = gr.Chatbot(height=450, label='HR Assistant')
    msg = gr.Textbox(placeholder='Ask a question about company policy...', label='You')
    clear = gr.Button('Clear Chat')

    msg.submit(chat, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: [], outputs=chatbot)

demo.launch(theme=gr.themes.Soft())

## Summary

| Step | What it does |
|------|--------------|
| Rephrase prompt | Converts follow-up questions into standalone questions |
| Retriever | Finds the most relevant document chunks for the question |
| Answer prompt | Uses context to generate a grounded response |
| Chat history | Passed as `HumanMessage` / `AIMessage` to the rephrase step |

**Key insight:** The rephrase step is what makes conversational RAG work — without it, follow-up questions like *"what about that?"* would retrieve wrong chunks.